# Ridership Data Processing

**Status: Active**

This notebook processes raw LA Metro ridership CSVs (received from LA Metro or via public
records request) and merges them into `src/data/ridership.json` and
`src/data/metro_line_metadata_current.json`.

For production runs, prefer `scripts/process_ridership.py` instead of this notebook.
Use this notebook for exploration, debugging, or one-off data investigations.

## Input CSV format
```
Year, Month, Line, DayType, Riders, Shakeup, Provider, Mode, Days
```
Where `DayType` is `DX` (weekday), `SA` (Saturday), or `SU` (Sunday).

## File paths
Update these paths relative to this notebook (run from repo root or adjust accordingly):
- Input CSV: `../data/raw/<file>.csv.gz`
- Ridership output: `../src/data/ridership.json`
- Line metadata output: `../src/data/metro_line_metadata_current.json`

In [ ]:
import pandas as pd
#import metro_ridership_scraper as scraper
import json
import numpy as np
from datetime import datetime

pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 1000)

In [ ]:
today = datetime.now().strftime('%Y-%m-%d')

## functions

In [ ]:
def convert_to_nested_json(df):
    print('converting data to nested json')
    df_json = df.groupby(
        ['year', 'month', 'line_name']
    )[[
        'est_wkday_ridership',
        'est_sat_ridership',
        'est_sun_ridership'
    ]].apply(
        lambda x: x.to_dict('records')
    ).reset_index(
        name='data'
    )
    df_json['new'] = df_json.data.str[0]
    df_json.line_name = df_json.line_name.astype(int)
    print(df_json.dtypes)
    json = df_json.groupby(
            ['year', 'month']
        )[[
            'line_name','new'
        ]].apply(
            lambda x: x.set_index('line_name')['new'].to_dict()
        ).reset_index(
            name='data'
    ).groupby(
            ['year']
        )[[
            'month','data'
        ]].apply(
            lambda x: x.set_index('month')['data'].to_dict()
    ).to_json(indent = 2)

    return json

def get_line_metadata(df):
    df = df.copy().sort_values(
        by=['year', 'month', 'line', 'daytype'],
        ascending=[False, True, True, True]
    )
    line_meta = df[[
        'line', 'mode', 'provider'
    ]].drop_duplicates(
        subset=['line', 'mode']
    ).sort_values(
        by=['mode', 'line']
    ).reset_index(drop=True)
    
    return line_meta

def get_monthly_metadata(df):
    monthly_meta = df[[
        'year', 'month', 'shakeup',
        'daytype', 'days'
    ]].copy().drop_duplicates().sort_values(
        by=['year', 'month', 'shakeup']
    ).reset_index(drop=True)

    shakeup = monthly_meta.groupby(by = [
        'year', 'month', 'daytype'
    ])['shakeup'].count().reset_index(drop = False)

    monthly_meta = monthly_meta.merge(
        shakeup,
        on = [
            'year', 'month', 'daytype'
        ],
        how = 'left',
        suffixes = ('', '_count')
    )
    days_sum = monthly_meta.groupby(by = [
        'year', 'month', 'daytype'
    ])[['days', 'shakeup_count']].sum().reset_index(drop = False)

    days_pivot = days_sum.pivot_table(
        values=['days'],
        index=['year','month','shakeup_count'],
        columns=['daytype']
    ).reset_index(
        drop=False, 
        col_level=1
    )
    days_pivot.columns = days_pivot.columns.droplevel()
    days_pivot = days_pivot.rename_axis(None, axis=1)
    days_pivot.rename(
        columns = {
            'year': 'year',
            'month': 'month',
            'DX':  'weekday', 
            'SA':  'saturday', 
            'SU':  'sunday'
        },
        inplace = True
    )
    days_pivot = days_pivot.merge(
        monthly_meta[[
            'year','month','shakeup'
        ]].copy().drop_duplicates(
            subset = ['year','month'],
            keep='last'
        ),
        on = ['year', 'month'],
        how = 'left'
    )
    days_pivot['shakeupid'] = np.nan
    days_pivot.loc[
        days_pivot.shakeup_count==4,
        'shakeupid'
    ] = days_pivot.shakeup

    days_meta = days_pivot[[
        'year', 'month', 'shakeupid', 
        'weekday', 'saturday', 'sunday']].copy()
    
    return days_meta
    
def get_monthly_nested(df):
    df_json = df.groupby(
        ['year', 'month']
    )[[
        'shakeupid', 
        'weekday', 'saturday', 'sunday'
    ]].apply(
        lambda x: x.to_dict('records')
    ).reset_index(
        name='data'
    )
    df_json['new'] = df_json.data.str[0]
    json_meta_nested = df_json.groupby(
            ['year']
        )[[
            'month','data'
        ]].apply(
            lambda x: x.set_index('month')['data'].to_dict()
    ).to_json(indent = 2)
    
    return json_meta_nested

## get new data

change to the proper file name/location when we have the system set up

In [ ]:
new_df = pd.read_csv('monthly_Riders_3.csv')
new_df.columns = new_df.columns.str.lower()
new_df.line = new_df.line.astype('int')

# recalculate average ridership accomodating for shakeups
# it looks like metro people round these numbers up (hence +0.5)
wm = lambda x: int(np.average(x+0.5, weights=new_df.loc[x.index, 'days']))
adjusted_df = new_df.groupby(
    ['year','month', 'line', 'daytype', 'provider', 'mode']
).agg(riders_weighted=('riders', wm)).reset_index(drop=False)

adjusted_full_df = adjusted_df.pivot_table(
    values=['riders_weighted'],
    index=['year','month', 'line', 'provider', 'mode'],
    columns=['daytype']
).reset_index(drop=False, col_level=1)
adjusted_full_df.columns = adjusted_full_df.columns.droplevel()
adjusted_full_df.rename(
    columns = {
        'year': 'year',
        'month': 'month', 
        'line': 'line_name',
        'DX':  'est_wkday_ridership', 
        'SA':  'est_sat_ridership', 
        'SU':  'est_sun_ridership'
    },
    inplace = True
)
adjusted_full_df = adjusted_full_df.rename_axis(None, axis=1)

metro_new_df = adjusted_full_df[[
    'year', 'month', 'line_name',
  'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
]].copy()

# deal with missing months for some lines
unique_ym = metro_new_df[['year', 'month']].drop_duplicates()
unique_line = metro_new_df[['line_name']].drop_duplicates()
calendar = unique_ym.merge(
    unique_line, 
    how='cross'
).sort_values(['year', 'month', 'line_name']).reset_index(drop=True)

metro_new_df = calendar.merge(
    metro_new_df, 
    on=['year', 'month', 'line_name'],
    how='left'
).fillna(0)

display(metro_new_df.head())
# metro_new_df.line_name = metro_new_df.line_name.astype('int')
full_metro_new_df.to_csv(f'metro_data_new_{today}.csv', index=False)

json_nested = convert_to_nested_json(metro_new_df)
with open(f'metro_ridership_nested_new.json', 'w') as f:
    f.write(json_nested)

json_flat = metro_new_df.to_json(orient = 'records', indent = 2)
with open(f'metro_ridership_flat_new.json', 'w') as f:
     f.write(json_flat)

## get metadata

In [ ]:
# line metadata
line_meta = get_line_metadata(new_df)
with open(f'metro_line_metadata_new.json', 'w') as f:
    f.write(line_meta.to_json(orient = 'records', indent = 2))

# monthly metadata
monthly_meta_df = get_monthly_metadata(new_df)
json_meta_nested = get_monthly_nested(monthly_meta_df)
with open(f'metro_monthly_metadata_new.json', 'w') as f:
    f.write(json_meta_nested)

## compare with existing data

compare with current files and update them if needed

### ridership data

In [ ]:
# if we decide not to store ridership data as csv
# with open(f'metro_ridership_current.json', 'r') as f:
#     current_json = json.load(f)
# 
# current_df = pd.DataFrame(
#     [
#         (
#             year, month, line,
#             info['est_wkday_ridership'],
#             info['est_sat_ridership'],
#             info['est_sun_ridership']
#         )
#         for year, months in current_json.items()
#         for month, lines in months.items()
#         for line, info in lines.items()
#     ],
#     columns=[
#     'year', 'month', 'line_name',
#     'est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership'
#     ]
# )
# display(current_df.head())

In [ ]:
current_df = pd.read_csv('metro_data_current.csv')

In [ ]:
# if newer data is missing some previously existing data,
# we'll fill in the blanks using previous data
metro_new_df_filled = metro_new_df.merge(
    current_df,
    on=['year', 'month', 'line_name'],
    how='left',
    suffixes=('_new', '_old')
)

for col in ['est_wkday_ridership', 'est_sat_ridership', 'est_sun_ridership']:
    metro_new_df_filled.loc[
        (metro_new_df_filled[col+'_new'].isnull()) &
        (~metro_new_df_filled[col+'_old'].isnull()),
        col+'_new'
    ] = metro_new_df_filled[col+'_old']

metro_new_df_filled.columns = metro_new_df_filled.columns.str.replace(
    '_new', ''
)
metro_new_df_filled = metro_new_df_filled[current_df.columns]

In [ ]:
final_df = pd.concat(
    [metro_new_df_filled, current_df]
).drop_duplicates(
    subset = ['year', 'month', 'line_name'],
    keep = 'first'
).sort_values(
    by=['year', 'month', 'line_name']
).reset_index(drop=True)

In [ ]:
updated = 0
if final_df.equals(current_df):
    print('ridership no updates')   
else:
    updated = 1
    final_df.to_csv('metro_data_current.csv', index=False)
    json_nested = convert_to_nested_json(final_df)
    with open(f'metro_ridership_nested_current.json', 'w') as f:
        f.write(json_nested)

    json_flat = final_df.to_json(orient = 'records', indent = 2)
    with open(f'metro_ridership_flat_current.json', 'w') as f:
         f.write(json_flat)
    print('ridership update saved')

### metadata

In [ ]:
if updated == 1:
    # line meta update
    with open(f'metro_line_metadata_current.json', 'r') as f:
         current_line_meta = pd.read_json(f)
    final_meta = pd.concat(
        [line_meta, current_line_meta]
    ).drop_duplicates(
        subset=['line', 'mode'], keep = 'first'
    ).sort_values(
        by=['line', 'mode']
    ).reset_index(drop=True)
    if final_meta.shape[0]>current_line_meta.shape[0]:
        with open(f'metro_line_metadata_current.json', 'w') as f:
             f.write(final_meta.to_json(orient = 'records', indent = 2))
        print('line meta update saved')
    else:
        print('line meta no updates')
    
    # monthly meta update
    with open(f'metro_monthly_metadata_current.json', 'r') as f:
        current_monthly_meta_json = json.load(f)
    current_monthly_meta_df = pd.DataFrame(
        [
            (
                year, month,
                info[0]['shakeupid'],
                info[0]['weekday'],
                info[0]['saturday'],
                info[0]['sunday']
            )
            for year, months in current_monthly_meta_json.items()
            for month, info in months.items()
        ],
        columns=[
        'year', 'month', 'shakeupid',
        'weekday', 'saturday', 'sunday'
        ]
    )
    monthly_meta_df.year = monthly_meta_df.year.astype(str)
    monthly_meta_df.month = monthly_meta_df.month.astype(str)
    final_monthly_meta_df = pd.concat(
            [monthly_meta_df, current_monthly_meta_df]
        ).drop_duplicates(
            subset = ['year', 'month', 'shakeupid'],
            keep = 'first'
        ).sort_values(
            by=['year', 'month']
        ).reset_index(drop=True)
    if final_monthly_meta_df.shape[0]>current_monthly_meta_df.shape[0]:
        final_meta_month_nested = get_monthly_nested(final_monthly_meta_df)
        with open(f'metro_monthly_metadata_current.json', 'w') as f:
             f.write(final_meta_month_nested)
        print('monthly meta update saved')
    else:
        print('monthly meta no updates')

## compare data from metro with scraped data

In [ ]:
# df_scraped = pd.read_csv(f'metro_ridership_scraped_2024_05_10.csv')
# df_scraped = df_scraped[df_scraped.line_name!='All'].copy()
# 
# compare_df = metro_df.merge(
#     dfs_scraped, 
#     on = ['year', 'month', 'line_name'],
#     how = 'outer',
#     suffixes = ('_metro', '_scraped')
# )
# 
# compare_df['wkday_delta'] = compare_df.est_wkday_ridership_metro - \
#     compare_df.est_wkday_ridership_scraped
# compare_df['sat_delta'] = compare_df.est_sat_ridership_metro - \
#     compare_df.est_sat_ridership_scraped
# compare_df['sun_delta'] = compare_df.est_sun_ridership_metro - \
#     compare_df.est_sun_ridership_scraped
# 
# compare_df[
#     (
#         (
#             (compare_df.est_wkday_ridership_metro!=
#              compare_df.est_wkday_ridership_scraped)&
#             compare_df.est_wkday_ridership_metro.notnull()&
#             compare_df.est_wkday_ridership_scraped.notnull()
#         )|
#         (
#             (compare_df.est_sat_ridership_metro!=
#              compare_df.est_sat_ridership_scraped)&
#             compare_df.est_sat_ridership_metro.notnull()&
#             compare_df.est_sat_ridership_scraped.notnull()
#         )|
#         (
#             (compare_df.est_sun_ridership_metro!=
#              compare_df.est_sun_ridership_scraped)&
#             compare_df.est_sun_ridership_metro.notnull()&
#             compare_df.est_sun_ridership_scraped.notnull()
#         )
#     )&(
#         (compare_df.wkday_delta!=0)|
#         (compare_df.sat_delta!=0)|
#         (compare_df.sun_delta!=0)
#     )
# ]

In [ ]:
# compare_df[
#     compare_df.est_wkday_ridership_metro.isnull()&
#     compare_df.est_sat_ridership_metro.isnull()&
#     compare_df.est_sun_ridership_metro.isnull()
# ]